In [0]:
import pyspark
from pyspark.sql.functions import *

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

df_bronze = spark.read.table("workspace.jamal.bronze_customers")

# ── Step 1: Cast correct types
df = df_bronze \
    .withColumn("customer_zip_code_prefix",
                F.col("customer_zip_code_prefix").cast(IntegerType()))

# ── Step 2: Normalise text — trim whitespace, lowercase city names
df = df \
    .withColumn("customer_city",
                F.lower(F.trim(F.col("customer_city")))) \
    .withColumn("customer_state",
                F.upper(F.trim(F.col("customer_state"))))

# ── Step 3: Drop rows missing critical identifiers
df = df.dropna(subset=["customer_id", "customer_unique_id"])

# ── Step 4: Remove exact duplicates on customer_id
df_silver = df.dropDuplicates(["customer_id"])

# ── Step 5: Add silver audit column
df_silver = df_silver.withColumn("_cleaned_at", F.current_timestamp())

# ── Step 6: Write Silver Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.jamal.silver_customers")

# ── Step 7: Quick quality check
print("Bronze row count:", df_bronze.count())
print("Silver row count:", df_silver.count())
display(spark.sql("SELECT customer_state, COUNT(*) as n FROM workspace.jamal.silver_customers GROUP BY 1 ORDER BY 2 DESC LIMIT 10"))